In [3]:
#########################
# 0. 환경 설정
!pip install onnx onnxruntime onnxscript

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 14.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#########################
# 1. ONNX 런타임 실행
import time
import torch
import onnxruntime as ort
from PIL import Image
from torchvision import models
from torchvision import transforms


def to_numpy(tensor):
    return tensor.detach().cpu().numpy()


image = Image.open("/content/drive/MyDrive/datasets/images/cat.jpg")
transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.48235, 0.45882, 0.40784],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)
input = transform(image).unsqueeze(0)

model = models.vgg16(num_classes=2)
#model.load_state_dict(torch.load("/content/drive/MyDrive/models/VGG16.pt"))
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/models/VGG16.pt",
        map_location=torch.device('cpu')
    )
)
model.eval()

with torch.no_grad():
    start_time = time.time()
    output = model(input)
    end_time = time.time()
    print("파이토치:")
    print(output)
    print(end_time - start_time)


ort_session = ort.InferenceSession("/content/drive/MyDrive/models/VGG16.onnx")

start_time = time.time()
ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(input)}
ort_outs = ort_session.run(output_names=None, input_feed=ort_inputs)
end_time = time.time()
print("ONNX:")
print(ort_outs)
print(end_time - start_time)

파이토치:
tensor([[ 6.8124, -7.9011]])
1.086822271347046
ONNX:
[array([[ 6.8124485, -7.901116 ]], dtype=float32)]
0.471635103225708
